# Granger causality

Fig. 2(b), left. Lagged platform-labeled supply is tested against the first-day median exposure of platform-labeled videos uploaded that day. Alternative lags, the reverse order, and the unlabeled-supply pair are in `analysis/robustness/timeseries/granger.ipynb`.

### 1. The three daily series

`first_day_median_exposure` is M_t. `labeled_supply` is s_t^a. `unlabeled_supply` is s_t^u. First-day exposure keeps later days of the same video out of the outcome.

In [1]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

import pandas as pd
from core.columns import FIRST_DAY_MEDIAN_EXPOSURE, LABELED_SUPPLY, UNLABELED_SUPPLY
from core.pipeline import ensure_synthetic, load_config

cfg = load_config(root)
daily = ensure_synthetic(root, cfg, rebuild=False)["daily_panel"]
series = daily[[FIRST_DAY_MEDIAN_EXPOSURE, LABELED_SUPPLY, UNLABELED_SUPPLY]]
print(len(series), "days")
series.head(8)


180 days


,first_day_median_exposure,labeled_supply,unlabeled_supply
0,35.000000,260.000000,894.285721
1,35.000000,249.476859,899.379997
2,37.067499,249.702765,900.793579
3,38.399524,226.287062,900.525169
4,40.993136,222.256079,897.671212
5,42.889274,220.198807,905.886064
6,42.533402,238.781954,910.630683
7,41.827825,228.877460,920.188495


### 2. First differences

The equation is fit on changes. A level trend does not enter the regression.

In [2]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

changes = series.diff().dropna()
print(len(changes), "changes")
changes.head(8)


179 changes


,first_day_median_exposure,labeled_supply,unlabeled_supply
1,0.000000,-10.523141,5.094276
2,2.067499,0.225907,1.413582
3,1.332024,-23.415703,-0.268410
4,2.593613,-4.030983,-2.853957
5,1.896138,-2.057272,8.214852
6,-0.355872,18.583147,4.744620
7,-0.705577,-9.904494,9.557812
8,-0.766656,-2.964744,-0.461899


### 3. Lags in the baseline equation

The baseline uses three lags. `focal_lag_*` are the coefficients on lagged labeled supply. Lagged unlabeled supply stays in the equation as the background supply. Weekday, month, holiday, campaign, and log vertical exposure are the controls.

In [3]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

from core.granger import granger_design

design = granger_design(
    daily,
    FIRST_DAY_MEDIAN_EXPOSURE,
    LABELED_SUPPLY,
    UNLABELED_SUPPLY,
    cfg["time_exog"],
    cfg["granger_lag"],
)
lag_columns = [column for column in design.columns if "_lag_" in column]
print(len(design), "estimation rows;", cfg["granger_lag"], "lags")
design[lag_columns].head(8)


176 estimation rows; 3 lags


,outcome_lag_1,focal_lag_1,comparison_lag_1,outcome_lag_2,focal_lag_2,comparison_lag_2,outcome_lag_3,focal_lag_3,comparison_lag_3
0,1.332024,-23.415703,-0.268410,2.067499,0.225907,1.413582,0.000000,-10.523141,5.094276
1,2.593613,-4.030983,-2.853957,1.332024,-23.415703,-0.268410,2.067499,0.225907,1.413582
2,1.896138,-2.057272,8.214852,2.593613,-4.030983,-2.853957,1.332024,-23.415703,-0.268410
3,-0.355872,18.583147,4.744620,1.896138,-2.057272,8.214852,2.593613,-4.030983,-2.853957
4,-0.705577,-9.904494,9.557812,-0.355872,18.583147,4.744620,1.896138,-2.057272,8.214852
5,-0.766656,-2.964744,-0.461899,-0.705577,-9.904494,9.557812,-0.355872,18.583147,4.744620
6,1.660438,32.175452,-0.011537,-0.766656,-2.964744,-0.461899,-0.705577,-9.904494,9.557812
7,-2.258420,22.321551,-0.897297,1.660438,32.175452,-0.011537,-0.766656,-2.964744,-0.461899


### 4. Coefficients and the joint test

The null is that the three labeled-supply lag coefficients are zero. The p-value is a residual moving-block bootstrap under that restricted model. Intervals use Newey-West standard errors.

In [4]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

from core.pipeline import run_granger_notebook

run_granger_notebook()


,term,estimate,standard_error,lag,f_statistic,bootstrap_p,bootstrap_draws
0,focal_lag_1,-0.082097,0.007585,3,48.424638,0.004975,200
1,focal_lag_2,-0.044740,0.008766,3,48.424638,0.004975,200
2,focal_lag_3,0.006175,0.007582,3,48.424638,0.004975,200
